# 02 — Làm sạch dữ liệu (Data Cleaning)

**Mục tiêu:** áp các quyết định từ [01_EDA.ipynb](./01_EDA.ipynb) lên dữ liệu thô để sinh ra `clean_data_train.csv` và `clean_data_test.csv` — sạch về mặt ngữ nghĩa (không còn junk), nhưng **chưa làm feature engineering** (việc đó ở stage 3).

**Quy tắc nguyên tắc của bước này:**

1. **Không leak:** mọi tham số (junk bucket exp, top-50 ngành…) đều được **fit trên train**, sau đó áp dụng nguyên xi cho test.
2. **Không impute statistic:** giá trị thiếu để `NaN` — stage 3 (feature engineering) sẽ quyết định cách impute phù hợp.
3. **Drop có chủ đích, có log:** mỗi bước drop dòng đều in `n_before / n_after / % giảm` để dễ kiểm chứng.

**Input:** `../raw_data/raw_data_{train,test}.csv`
**Output:** `../clean_data/clean_data_{train,test}.csv` — 14 cột.

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 140)

ROOT = Path.cwd().parent
RAW_DIR = ROOT / 'raw_data'
CLEAN_DIR = ROOT / 'clean_data'
CLEAN_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
USD_TO_VND = 25_000
SALARY_CAP_MILLION = 100       # drop dòng có expected_salary > 100 triệu (junk)
TOP_K_INDUSTRY = 50            # giữ top-50 ngành base, còn lại gom vào 'Other'
EXP_JUNK_MIN_N = 500           # bucket có < 500 dòng & tỉ lệ position missing cao → junk
EXP_JUNK_POS_RATE = 0.5

## 1. Đọc dữ liệu thô

In [2]:
train_raw = pd.read_csv(RAW_DIR / 'raw_data_train.csv')
test_raw = pd.read_csv(RAW_DIR / 'raw_data_test.csv')

print(f"Raw train: {train_raw.shape}")
print(f"Raw test : {test_raw.shape}")
print(f"Columns  : {list(train_raw.columns)}")

train = train_raw.copy()
test = test_raw.copy()

Raw train: (485502, 14)
Raw test : (121376, 14)
Columns  : ['id', 'job_title', 'company_name', 'salary', 'location', 'job_type', 'job_industry', 'experience_level', 'education_level', 'job_position', 'job_description', 'benefits', 'requirements', 'year']


## 2. Parse target — `expected_salary` (triệu VNĐ)

Áp các pattern đã chốt ở EDA Section 2.2. Sau khi parse:
- Drop dòng không parse được (≈ 5%: 'thoả thuận', '0 VND', 'Đang cập nhật'…)
- Drop dòng có `expected_salary > 100 triệu` (junk; xem EDA Section 2.5).

In [3]:
PATTERNS = {
    'vnd_range_dotted':  re.compile(r'^([\d.]+)\s*-\s*([\d.]+)\s*VND$'),
    'vnd_range_month':   re.compile(r'^(\d+)\s*-\s*(\d+)\s*VND MONTH$'),
    'vnd_single_dotted': re.compile(r'^([\d.]+)\s*VND$'),
    'usd_range':         re.compile(r'\$?\s*(\d+)\s*-\s*\$?\s*(\d+)\s*USD', re.IGNORECASE),
}


def parse_to_million(s):
    if not isinstance(s, str):
        return np.nan

    m = PATTERNS['vnd_range_dotted'].search(s)
    if m:
        lo, hi = int(m.group(1).replace('.', '')), int(m.group(2).replace('.', ''))
        return (lo + hi) / 2 / 1e6 if (lo or hi) else np.nan

    m = PATTERNS['vnd_range_month'].search(s)
    if m:
        lo, hi = int(m.group(1)), int(m.group(2))
        return (lo + hi) / 2 / 1e6 if (lo or hi) else np.nan

    m = PATTERNS['vnd_single_dotted'].search(s)
    if m:
        v = int(m.group(1).replace('.', ''))
        return v / 1e6 if v > 0 else np.nan

    m = PATTERNS['usd_range'].search(s)
    if m:
        lo, hi = int(m.group(1)), int(m.group(2))
        return (lo + hi) / 2 * USD_TO_VND / 1e6

    return np.nan


train['expected_salary'] = train['salary'].map(parse_to_million)
test['expected_salary'] = test['salary'].map(parse_to_million)

In [4]:
def salary_diagnostic(df, name):
    n = len(df)
    parsed = df['expected_salary'].notna().sum()
    high = (df['expected_salary'] > SALARY_CAP_MILLION).sum()
    print(f"{name:5s}: n={n:>7,} | parsed={parsed:>7,} ({parsed/n*100:.2f}%) | "
          f"> {SALARY_CAP_MILLION}M = {high:>4,} ({high/n*100:.3f}%)")


print("Trước khi drop:")
salary_diagnostic(train, 'train')
salary_diagnostic(test, 'test')

Trước khi drop:
train: n=485,502 | parsed=459,210 (94.58%) | > 100M =  642 (0.132%)
test : n=121,376 | parsed=114,923 (94.68%) | > 100M =  150 (0.124%)


In [5]:
def filter_salary(df, name):
    n0 = len(df)
    df = df.dropna(subset=['expected_salary']).copy()
    n1 = len(df)
    df = df[df['expected_salary'] <= SALARY_CAP_MILLION].copy()
    n2 = len(df)
    print(f"{name:5s}: {n0:>7,} → drop NaN: {n0-n1:>6,} → drop >{SALARY_CAP_MILLION}M: {n1-n2:>5,} → còn {n2:>7,} ({n2/n0*100:.2f}%)")
    return df


train = filter_salary(train, 'train')
test = filter_salary(test, 'test')

print(f"\nThống kê `expected_salary` (train, triệu VNĐ):")
print(train['expected_salary'].describe().round(2).to_string())

train: 485,502 → drop NaN: 26,292 → drop >100M:   642 → còn 458,568 (94.45%)
test : 121,376 → drop NaN:  6,453 → drop >100M:   150 → còn 114,773 (94.56%)

Thống kê `expected_salary` (train, triệu VNĐ):
count    458568.00
mean         13.32
std           6.80
min           0.15
25%           9.00
50%          12.00
75%          15.00
max         100.00


### Trực quan — phân phối `expected_salary` trước/sau drop

So sánh phân phối trước (parsed all) và sau (drop NaN + cap 100M) để biện minh ngưỡng cắt: phần đuôi > 100tr toàn là sentinel `-999.000.000 VND` (giá trị tối đa giả định), không phải dữ liệu thật.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'

salary_before = train_raw['salary'].map(parse_to_million).dropna()
salary_after = train['expected_salary']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.hist(salary_before, bins=120, range=(0, 1200), color='steelblue', alpha=0.85, edgecolor='white', linewidth=0.3)
ax.axvline(SALARY_CAP_MILLION, color='red', linestyle='--', lw=1.2, label=f'Ngưỡng drop = {SALARY_CAP_MILLION}tr')
ax.set_yscale('log')
ax.set_xlabel('expected_salary (triệu VNĐ)')
ax.set_ylabel('Số JD (log scale)')
ax.set_title(f'Trước khi drop  (n={len(salary_before):,}, max={salary_before.max():.0f}tr)')
ax.legend()

ax = axes[1]
ax.hist(salary_after, bins=80, range=(0, 100), color='seagreen', alpha=0.85, edgecolor='white', linewidth=0.3)
ax.set_xlabel('expected_salary (triệu VNĐ)')
ax.set_ylabel('Số JD')
ax.set_title(f'Sau khi drop  (n={len(salary_after):,}, max={salary_after.max():.0f}tr)')
ax.set_xlim(0, 100)

plt.suptitle('Phân phối lương — trước và sau khi drop NaN + cap 100tr (train)', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cleaning_salary_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'cleaning_salary_distribution.png'}")

## 3. Trích `years_exp` từ `experience_level`

Regex `(\d+)\s*năm`. Bucket junk được phát hiện bằng luật (xem EDA Section 7.2.1):
> `n_rows < 500` VÀ `pos_missing_rate > 0.5` → coi là junk → set `years_exp = NaN`.

Tham số junk được **fit trên train**, áp dụng nguyên cho test.

In [6]:
def extract_years(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip()
    if 'Dưới 1' in s:
        return 0.5
    if s in {'Không', 'Chưa cập nhật', 'Không yêu cầu'}:
        return np.nan
    m = re.search(r'(\d+)\s*năm', s)
    return float(m.group(1)) if m else np.nan


train['years_exp'] = train['experience_level'].map(extract_years)
test['years_exp'] = test['experience_level'].map(extract_years)

In [7]:
df_diag = train.dropna(subset=['years_exp']).copy()
df_diag['_pos_missing'] = (df_diag['job_position'] == 'Chưa cập nhật')

exp_stats = df_diag.groupby('years_exp').agg(
    n_rows=('id', 'size'),
    median_salary=('expected_salary', 'median'),
    pos_missing_rate=('_pos_missing', 'mean'),
).round(3)

junk_buckets = exp_stats[
    (exp_stats['n_rows'] < EXP_JUNK_MIN_N)
    & (exp_stats['pos_missing_rate'] > EXP_JUNK_POS_RATE)
].index.tolist()

print("Cấu trúc theo years_exp (train):")
print(exp_stats.to_string())
print(f"\nBucket bị nhận diện là junk (fit trên train): {junk_buckets}")

Cấu trúc theo years_exp (train):
           n_rows  median_salary  pos_missing_rate
years_exp                                         
0.5          4912          10.00             0.265
1.0        126527          10.50             0.100
2.0         72586          11.00             0.081
3.0        138248          11.75             0.067
4.0         65325          13.50             0.059
5.0         29846          16.00             0.061
6.0          2125          17.50             0.052
7.0          7196          19.00             0.069
8.0          2411          22.50             0.072
9.0           225           8.50             0.996

Bucket bị nhận diện là junk (fit trên train): [9.0]


In [8]:
for df, name in [(train, 'train'), (test, 'test')]:
    n_before = df['years_exp'].notna().sum()
    df.loc[df['years_exp'].isin(junk_buckets), 'years_exp'] = np.nan
    n_after = df['years_exp'].notna().sum()
    print(f"{name:5s}: years_exp non-null {n_before:>7,} → {n_after:>7,} (giảm {n_before-n_after})")

print(f"\nPhân phối years_exp sau lọc (train):")
print(train['years_exp'].value_counts().sort_index().to_string())

train: years_exp non-null 449,401 → 449,176 (giảm 225)
test : years_exp non-null 112,458 → 112,409 (giảm 49)

Phân phối years_exp sau lọc (train):
years_exp
0.5      4912
1.0    126527
2.0     72586
3.0    138248
4.0     65325
5.0     29846
6.0      2125
7.0      7196
8.0      2411


### Trực quan — bucket `years_exp` & nhận diện junk

Bar chart số lượng + median lương theo bucket năm KN, **highlight đỏ** bucket bị rule-based nhận diện là junk (`n<500` & `pos_missing>0.5`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

buckets = exp_stats.index.astype(str).tolist()
colors = ['tab:red' if float(b) in junk_buckets else 'steelblue' for b in buckets]

ax = axes[0]
ax.bar(buckets, exp_stats['n_rows'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_yscale('log')
ax.set_xlabel('years_exp bucket')
ax.set_ylabel('Số JD (log scale)')
ax.set_title('Số JD theo bucket years_exp (train)')
for i, (b, n) in enumerate(zip(buckets, exp_stats['n_rows'])):
    ax.text(i, n * 1.15, f'{n:,}', ha='center', fontsize=8)
# annotation cho junk
junk_idx = [i for i, b in enumerate(buckets) if float(b) in junk_buckets]
for i in junk_idx:
    ax.annotate(
        f'JUNK\n(pos_missing={exp_stats["pos_missing_rate"].iloc[i]:.1%})',
        xy=(i, exp_stats['n_rows'].iloc[i]),
        xytext=(i, exp_stats['n_rows'].iloc[i] * 30),
        ha='center', fontsize=8, color='red',
        arrowprops=dict(arrowstyle='->', color='red', lw=1),
    )

ax = axes[1]
ax.bar(buckets, exp_stats['median_salary'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('years_exp bucket')
ax.set_ylabel('Median lương (triệu)')
ax.set_title('Median lương theo bucket years_exp')
for i, (b, m) in enumerate(zip(buckets, exp_stats['median_salary'])):
    ax.text(i, m + 0.3, f'{m:.1f}', ha='center', fontsize=8)

plt.suptitle('years_exp — nhận diện junk bucket bằng rule (n<500 & pos_missing>0.5)', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cleaning_years_exp_buckets.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'cleaning_years_exp_buckets.png'}")

## 4. Trích `province` từ `location`

Regex 2 lớp:
- **Lớp 1:** tìm tên 63 tỉnh/thành trong chuỗi.
- **Lớp 2 (fallback):** tra tên quận/huyện của HCM và Hà Nội nếu lớp 1 fail.

Không match được → `'Other'`.

In [9]:
VN_PROVINCES = [
    'Hồ Chí Minh', 'Hà Nội', 'Đà Nẵng', 'Hải Phòng', 'Cần Thơ',
    'An Giang', 'Bà Rịa - Vũng Tàu', 'Bắc Giang', 'Bắc Kạn', 'Bạc Liêu',
    'Bắc Ninh', 'Bến Tre', 'Bình Định', 'Bình Dương', 'Bình Phước',
    'Bình Thuận', 'Cà Mau', 'Cao Bằng', 'Đắk Lắk', 'Đắk Nông',
    'Điện Biên', 'Đồng Nai', 'Đồng Tháp', 'Gia Lai', 'Hà Giang',
    'Hà Nam', 'Hà Tĩnh', 'Hải Dương', 'Hậu Giang', 'Hòa Bình',
    'Hưng Yên', 'Khánh Hòa', 'Kiên Giang', 'Kon Tum', 'Lai Châu',
    'Lâm Đồng', 'Lạng Sơn', 'Lào Cai', 'Long An', 'Nam Định',
    'Nghệ An', 'Ninh Bình', 'Ninh Thuận', 'Phú Thọ', 'Phú Yên',
    'Quảng Bình', 'Quảng Nam', 'Quảng Ngãi', 'Quảng Ninh', 'Quảng Trị',
    'Sóc Trăng', 'Sơn La', 'Tây Ninh', 'Thái Bình', 'Thái Nguyên',
    'Thanh Hóa', 'Thừa Thiên Huế', 'Tiền Giang', 'Trà Vinh', 'Tuyên Quang',
    'Vĩnh Long', 'Vĩnh Phúc', 'Yên Bái',
]

DISTRICT_TO_PROVINCE = {
    **{f'Quận {i}': 'Hồ Chí Minh' for i in range(1, 13)},
    'Tân Bình': 'Hồ Chí Minh', 'Bình Thạnh': 'Hồ Chí Minh',
    'Phú Nhuận': 'Hồ Chí Minh', 'Gò Vấp': 'Hồ Chí Minh',
    'Thủ Đức': 'Hồ Chí Minh', 'Hóc Môn': 'Hồ Chí Minh',
    'Củ Chi': 'Hồ Chí Minh', 'Nhà Bè': 'Hồ Chí Minh',
    'Bình Chánh': 'Hồ Chí Minh', 'Cần Giờ': 'Hồ Chí Minh',
    'Tân Phú': 'Hồ Chí Minh', 'Bình Tân': 'Hồ Chí Minh',
    'Hoàn Kiếm': 'Hà Nội', 'Ba Đình': 'Hà Nội', 'Đống Đa': 'Hà Nội',
    'Cầu Giấy': 'Hà Nội', 'Thanh Xuân': 'Hà Nội',
    'Hai Bà Trưng': 'Hà Nội', 'Tây Hồ': 'Hà Nội',
    'Long Biên': 'Hà Nội', 'Hà Đông': 'Hà Nội',
    'Nam Từ Liêm': 'Hà Nội', 'Bắc Từ Liêm': 'Hà Nội',
    'Hoàng Mai': 'Hà Nội',
}

_PROVINCE_RE = re.compile('|'.join(re.escape(p) for p in VN_PROVINCES), re.IGNORECASE)
_DISTRICT_RE = re.compile('|'.join(re.escape(d) for d in DISTRICT_TO_PROVINCE), re.IGNORECASE)
_LOOKUP = {p.lower(): p for p in VN_PROVINCES}
_LOOKUP.update({k.lower(): v for k, v in DISTRICT_TO_PROVINCE.items()})


def extract_province(s):
    if not isinstance(s, str):
        return 'Other'
    m = _PROVINCE_RE.search(s)
    if m:
        return _LOOKUP[m.group(0).lower()]
    m = _DISTRICT_RE.search(s)
    if m:
        return _LOOKUP[m.group(0).lower()]
    return 'Other'


train['province'] = train['location'].map(extract_province)
test['province'] = test['location'].map(extract_province)

In [10]:
for df, name in [(train, 'train'), (test, 'test')]:
    coverage = (df['province'] != 'Other').mean()
    print(f"{name:5s}: coverage {coverage:.2%} | unique = {df['province'].nunique()} | "
          f"Other = {(df['province'] == 'Other').sum():,}")

print(f"\nTop 10 province (train):")
print(train['province'].value_counts().head(10).to_string())

train: coverage 93.84% | unique = 64 | Other = 28,259
test : coverage 93.82% | unique = 64 | Other = 7,098

Top 10 province (train):
province
Hồ Chí Minh    205669
Hà Nội         118003
Other           28259
Bình Dương      21801
Đồng Nai         9145
Long An          7781
Đà Nẵng          6963
Điện Biên        6149
Bình Phước       5121
Bắc Ninh         4287


### Trực quan — top 15 tỉnh/thành (train)

Bar ngang top 15 + nhóm 'Other' (cell không match được regex 2 lớp). HCM + Hà Nội chiếm phần lớn — phản ánh thị trường tuyển dụng tập trung 2 thành phố.

In [ ]:
prov_counts = train['province'].value_counts()
top_n = 15
top = prov_counts.head(top_n)
other_count = prov_counts.iloc[top_n:].sum() + (prov_counts.get('Other', 0) - prov_counts.get('Other', 0))
# Đảm bảo 'Other' luôn xuất hiện ở cuối
if 'Other' not in top.index:
    top = pd.concat([top, pd.Series({'Other': prov_counts.get('Other', 0)})])

pct = top / len(train) * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors_p = ['tab:gray' if name == 'Other' else 'steelblue' for name in top.index]
y_pos = np.arange(len(top))[::-1]
ax.barh(y_pos, top.values, color=colors_p, edgecolor='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(top.index)
ax.set_xlabel('Số JD')
ax.set_title(f'Top {top_n} tỉnh/thành + Other  (coverage = {(train["province"] != "Other").mean()*100:.2f}%)')
for i, (n, p) in enumerate(zip(top.values, pct.values)):
    ax.text(n + max(top.values) * 0.005, y_pos[i], f'{n:,} ({p:.1f}%)', va='center', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / 'cleaning_province_top15.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'cleaning_province_top15.png'}")

## 5. `job_industry` → `industries_list` (multi-label)

Tách dấu `/` → list ngành base. Giữ top-50 ngành phổ biến nhất (fit trên train), còn lại gom vào `'Other'`. Lưu thành chuỗi nối bằng `|` cho gọn trong CSV.

In [11]:
def split_industries(s):
    if not isinstance(s, str):
        return []
    return [x.strip() for x in s.split('/') if x.strip()]


train_lists = train['job_industry'].map(split_industries)
all_base = pd.Series([ind for lst in train_lists for ind in lst])
top_k = all_base.value_counts().head(TOP_K_INDUSTRY)
TOP_INDUSTRIES = set(top_k.index)

print(f"Tổng số token base trên train: {len(all_base):,}")
print(f"Số ngành base unique         : {all_base.nunique():,}")
print(f"Top {TOP_K_INDUSTRY} ngành cover         : {top_k.sum() / len(all_base) * 100:.2f}%")
print(f"\n5 ngành phổ biến nhất:")
print(top_k.head(5).to_string())

Tổng số token base trên train: 629,570
Số ngành base unique         : 95
Top 50 ngành cover         : 99.97%

5 ngành phổ biến nhất:
Chăm sóc khách hàng      58327
Bán hàng - Kinh doanh    48874
Kế toán                  40869
Kiểm toán                40159
Giáo dục - Đào tạo       27349


In [12]:
def filter_industries(s):
    inds = split_industries(s)
    if not inds:
        return 'Other'
    kept = [ind if ind in TOP_INDUSTRIES else 'Other' for ind in inds]
    # khử trùng lặp, giữ thứ tự xuất hiện
    seen = []
    for x in kept:
        if x not in seen:
            seen.append(x)
    return '|'.join(seen)


train['industries_list'] = train['job_industry'].map(filter_industries)
test['industries_list'] = test['job_industry'].map(filter_industries)

n_industries_per_row = train['industries_list'].str.split('|').str.len()
print(f"Số ngành trên mỗi bài (train) — phân phối:")
print(n_industries_per_row.value_counts().sort_index().to_string())
print(f"\nTỉ lệ row chỉ có 'Other': {(train['industries_list'] == 'Other').mean():.2%}")

Số ngành trên mỗi bài (train) — phân phối:
industries_list
1    292967
2    160225
3      5376

Tỉ lệ row chỉ có 'Other': 0.02%


### Trực quan — top 20 ngành (multi-label, train)

Mỗi JD có 1–3 ngành (đã in distribution ở cell trên), nên tổng % > 100%. Bar chart % JD chứa ngành đó. So sánh thêm median lương theo ngành để thấy spread.

In [ ]:
ind_exploded = train.assign(_ind=train['industries_list'].str.split('|')).explode('_ind')
ind_counts = ind_exploded['_ind'].value_counts()
ind_median = ind_exploded.groupby('_ind')['expected_salary'].median()

top_n = 20
top_ind = ind_counts.head(top_n).index.tolist()
ind_pct = (ind_counts.loc[top_ind] / len(train) * 100).round(2)
ind_med = ind_median.loc[top_ind].round(2)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)

y_pos = np.arange(len(top_ind))[::-1]
colors_i = ['tab:gray' if name == 'Other' else 'steelblue' for name in top_ind]

ax = axes[0]
ax.barh(y_pos, ind_pct.values, color=colors_i, edgecolor='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_ind, fontsize=9)
ax.set_xlabel('% JD chứa ngành')
ax.set_title(f'Top {top_n} ngành (xuất hiện trong industries_list)')
for i, p in enumerate(ind_pct.values):
    ax.text(p + 0.1, y_pos[i], f'{p:.1f}%', va='center', fontsize=8)
ax.invert_yaxis()

ax = axes[1]
ax.barh(y_pos, ind_med.values, color='seagreen', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Median lương (triệu)')
ax.set_title('Median lương theo ngành')
for i, m in enumerate(ind_med.values):
    ax.text(m + 0.15, y_pos[i], f'{m:.1f}', va='center', fontsize=8)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(FIG_DIR / 'cleaning_industries_top20.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'cleaning_industries_top20.png'}")

## 6. Chuẩn hoá categorical & text

- **Categorical ngắn** (`education_level`, `job_type`, `job_position`): `lower().strip()`, giá trị thiếu → `'unknown'`.
- **Text dài** (`job_title`, `company_name`, `job_description`, `requirements`, `benefits`): `strip()`, NaN → `''`. Không lowercase ở đây — stage 3 sẽ lowercase trong vectorizer.

In [13]:
def normalize_cat(s):
    if not isinstance(s, str):
        return 'unknown'
    s = s.strip().lower()
    return s if s else 'unknown'


for col in ['education_level', 'job_type', 'job_position']:
    for df in (train, test):
        df[col] = df[col].map(normalize_cat)

for col in ['job_title', 'company_name', 'job_description', 'requirements', 'benefits']:
    for df in (train, test):
        df[col] = df[col].fillna('').astype(str).str.strip()

print("Cardinality sau normalize (train):")
for col in ['education_level', 'job_type', 'job_position']:
    print(f"  {col:18s}: {train[col].nunique():>3}")

print("\nTỉ lệ rỗng text (train):")
for col in ['job_title', 'company_name', 'job_description', 'requirements', 'benefits']:
    empty = (train[col] == '').mean()
    print(f"  {col:18s}: {empty*100:5.2f}%")

Cardinality sau normalize (train):
  education_level   :   7
  job_type          :   8
  job_position      :  15

Tỉ lệ rỗng text (train):
  job_title         :  0.00%
  company_name      :  0.00%
  job_description   :  0.00%
  requirements      :  0.02%
  benefits          :  0.00%


## 7. Loại trùng theo `id` và chốt schema

EDA cho thấy `id` không có trùng, đây chỉ là bước an toàn. Sau đó chốt 14 cột cuối.

In [14]:
for df, name in [(train, 'train'), (test, 'test')]:
    n0 = len(df)
    df.drop_duplicates(subset='id', inplace=True)
    print(f"{name:5s}: drop {n0 - len(df)} dòng trùng id → còn {len(df):,}")

train: drop 0 dòng trùng id → còn 458,568
test : drop 0 dòng trùng id → còn 114,773


In [15]:
FINAL_COLUMNS = [
    'id',
    'expected_salary',          # target
    'years_exp',
    'year',
    'province',
    'industries_list',
    'education_level',
    'job_type',
    'job_position',
    'job_title',
    'company_name',
    'job_description',
    'requirements',
    'benefits',
]

train_clean = train[FINAL_COLUMNS].copy()
test_clean = test[FINAL_COLUMNS].copy()
train_clean['expected_salary'] = train_clean['expected_salary'].round(2)
test_clean['expected_salary'] = test_clean['expected_salary'].round(2)

print("Schema clean_data_train:")
print(train_clean.dtypes.to_string())

print(f"\nMissing (%) — train:")
miss = (train_clean.isna().mean() * 100).round(3)
print(miss[miss > 0].to_string() or '  (không cột nào missing)')

print(f"\nShape : train={train_clean.shape}, test={test_clean.shape}")

Schema clean_data_train:
id                   int64
expected_salary    float64
years_exp          float64
year                 int64
province               str
industries_list        str
education_level        str
job_type               str
job_position           str
job_title              str
company_name           str
job_description        str
requirements           str
benefits               str

Missing (%) — train:
years_exp    2.048

Shape : train=(458568, 14), test=(114773, 14)


## 8. Lưu `clean_data_train.csv` và `clean_data_test.csv`

In [16]:
train_path = CLEAN_DIR / 'clean_data_train.csv'
test_path = CLEAN_DIR / 'clean_data_test.csv'

train_clean.to_csv(train_path, index=False, encoding='utf-8-sig')
test_clean.to_csv(test_path, index=False, encoding='utf-8-sig')

print(f"Đã lưu:")
print(f"  {train_path}")
print(f"    -> {train_path.stat().st_size / 1e6:7.1f} MB ({len(train_clean):,} dòng)")
print(f"  {test_path}")
print(f"    -> {test_path.stat().st_size  / 1e6:7.1f} MB ({len(test_clean):,} dòng)")

Đã lưu:
  D:\Documents\School\Ki_6\KHDL\09 - Dự đoán mức lương kỳ vọng dựa trên bản mô tả công việc (Job Description)\clean_data\clean_data_train.csv
    ->   845.4 MB (458,568 dòng)
  D:\Documents\School\Ki_6\KHDL\09 - Dự đoán mức lương kỳ vọng dựa trên bản mô tả công việc (Job Description)\clean_data\clean_data_test.csv
    ->   211.4 MB (114,773 dòng)


## 9. Tổng kết — trước và sau khi clean

In [17]:
summary = pd.DataFrame({
    'raw': {
        'n_train': len(train_raw),
        'n_test': len(test_raw),
        'province_unique_train': train_raw['location'].nunique(),
        'industry_unique_train': train_raw['job_industry'].nunique(),
        'salary_missing_train (%)': round(train_raw['salary'].isna().mean() * 100, 3),
    },
    'clean': {
        'n_train': len(train_clean),
        'n_test': len(test_clean),
        'province_unique_train': train_clean['province'].nunique(),
        'industry_unique_train': pd.Series([
            x for s in train_clean['industries_list'] for x in s.split('|')
        ]).nunique(),
        'salary_missing_train (%)': round(train_clean['expected_salary'].isna().mean() * 100, 3),
    },
})
print(summary.to_string())

retain_train = len(train_clean) / len(train_raw) * 100
retain_test = len(test_clean) / len(test_raw) * 100
print(f"\nGiữ lại: train={retain_train:.2f}%, test={retain_test:.2f}%")
print("\nMẫu 3 dòng clean_data_train:")
print(train_clean.head(3).to_string())

                               raw     clean
n_train                   485502.0  458568.0
n_test                    121376.0  114773.0
province_unique_train     215997.0      64.0
industry_unique_train       3830.0      51.0
salary_missing_train (%)       0.0       0.0

Giữ lại: train=94.45%, test=94.56%

Mẫu 3 dòng clean_data_train:
       id  expected_salary  years_exp  year     province                                                    industries_list education_level        job_type   job_position                                                job_title                                           company_name                                                                                                                                                                                                                                                                                                                                                                                              

## 10. Bước tiếp theo

Sang [03_Feature_Engineering.ipynb](./03_Feature_Engineering.ipynb):

1. Đọc `clean_data/clean_data_{train,test}.csv`.
2. **Numeric:** `years_exp`, `year` — impute (KNN/median) + có thể chuẩn hoá.
3. **Categorical low-card:** `education_level`, `job_type`, `job_position` — one-hot.
4. **Categorical mid-card:** `province` — one-hot 63 + Other.
5. **Multi-label:** `industries_list` — multi-hot top-50 + Other.
6. **Text:** `job_description`, `requirements`, `benefits` (và optional `job_title`) — TF-IDF riêng từng cột, hstack.
7. **Target:** `log1p(expected_salary)` cho modeling.
8. Output: ma trận sparse + vector `y` → notebook 04 modeling.